# 008 - XGBoost ajustado

## Objetivo

Aprofundar o XGBoost do 007, partindo de `max_depth=5` e `learning_rate=0.1`. Mantemos as mesmas variáveis, a divisão estratificada 80/20 (`random_state=42`) e a preparação comum sem normalização. A seleção e a análise de ablação usam os mesmos três folds estratificados apenas na parcela de treino; a validação fixa é usada uma vez para a avaliação final.

Na análise de sinalização, definimos previamente como caso sinalizado aquele cuja previsão out-of-fold difere de `AGE_RANGE` real. Isto mede erro de classificação, não constitui uma definição clínica ou verdade de anomalia.

## Ambiente, dados e divisão fixa

Reutilizamos `preparar_dados()`, `make_stratified_split()` e `make_model_pipeline()` de `src/preparar_dados.py`. A divisão e as variáveis são as mesmas do 006/007. A validação não participa na pesquisa de parâmetros nem no early stopping: em cada fold, uma parcela estratificada do treino interno serve apenas para early stopping, e a outra parte do fold mede Macro F1.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost

candidate_roots = [Path.cwd(), *Path.cwd().resolve().parents]
repo_root = next(
    (p for p in candidate_roots if (p / 'notebooks').is_dir() and (p / 'src' / 'preparar_dados.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto com notebooks/ e src/preparar_dados.py.')

sys.path.insert(0, str(repo_root))
from src.preparar_dados import make_model_pipeline, make_stratified_split, preparar_dados

train_path = repo_root / 'notebooks' / 'dataset_train.csv'
test_path = repo_root / 'notebooks' / 'dataset_test_X.csv'
if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError(f'CSV de treino/teste não encontrados em {repo_root / "notebooks"}')

print(f'Python: {sys.version}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'XGBoost: {xgboost.__version__}')
print(f'Raiz do projeto: {repo_root}')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
prepared = preparar_dados(train_df, test_df)
X_train, X_val, y_train, y_val = make_stratified_split(
    prepared['X_train'], prepared['y_train'], test_size=0.2, random_state=42
)

assert len(X_train) == 8769 and len(X_val) == 2193, (len(X_train), len(X_val))
assert X_train.columns.equals(X_val.columns), 'As colunas de treino e validação não estão alinhadas.'
assert sorted(y_train.unique().tolist()) == list(range(16)), 'Esperavam-se as 16 classes no treino.'
print(f'Treino: {X_train.shape}; validação: {X_val.shape}')
print(f'Classes no alvo: {sorted(y_train.unique().tolist())}')

Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
scikit-learn: 1.6.1
XGBoost: 3.4.1
Raiz do projeto: /Users/eduardaferreira/daa-kaggle-2026-2027


Treino: (8769, 236); validação: (2193, 236)
Classes no alvo: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


## Pesquisa com early stopping dentro de cada fold

Comparamos as oito combinações de `min_child_weight=[1, 5]`, `subsample=[0.8, 1.0]` e `colsample_bytree=[0.8, 1.0]`, mantendo `max_depth=5` e `learning_rate=0.1`. Cada ajuste pode usar até 500 árvores, com paciência de 30 rondas e `eval_metric='mlogloss'` para early stopping. Em cada fold, reservamos 15% do respetivo treino interno de forma estratificada para early stopping; o pré-processador ajusta-se apenas nos restantes 85%, que também transformam a parcela de early stopping e a validação do fold. A Macro F1 é medida exclusivamente na validação do fold. A mesma definição de folds é reutilizada na seleção e na análise OOF.

In [2]:
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from xgboost import XGBClassifier

MAX_ESTIMATORS = 500
EARLY_STOPPING_ROUNDS = 30
EARLY_STOPPING_METRIC = 'mlogloss'
EARLY_STOPPING_SIZE = 0.15
FIXED_XGB_PARAMS = {
    'objective': 'multi:softprob',
    'num_class': 16,
    'tree_method': 'hist',
    'n_estimators': MAX_ESTIMATORS,
    'max_depth': 5,
    'learning_rate': 0.1,
    'random_state': 42,
    'n_jobs': 1,
    'eval_metric': EARLY_STOPPING_METRIC,
    'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
    'verbosity': 0,
}
FOLD_SPLITS = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(X_train, y_train))

def run_xgb_fold_cv(X, y, fold_splits, parameter_grid, *, record_oof=False):
    """Fit prep on an inner fit partition, early-stop on its holdout, score on outer fold."""
    from sklearn.base import clone

    rows = []
    oof_predictions = np.full(len(X), -1, dtype=int) if record_oof else None
    for fold_number, (outer_train_pos, outer_val_pos) in enumerate(fold_splits, start=1):
        X_outer_train = X.iloc[outer_train_pos]
        y_outer_train = y.iloc[outer_train_pos]
        X_outer_val = X.iloc[outer_val_pos]
        y_outer_val = y.iloc[outer_val_pos]

        X_fit, X_early, y_fit, y_early = train_test_split(
            X_outer_train,
            y_outer_train,
            test_size=EARLY_STOPPING_SIZE,
            random_state=42,
            stratify=y_outer_train,
        )
        fold_pipe = make_model_pipeline(
            XGBClassifier(**{**FIXED_XGB_PARAMS, **parameter_grid}),
            normalize_numeric=False,
        )
        preprocessor = fold_pipe.named_steps['preprocessor']
        model = fold_pipe.named_steps['model']
        preprocessor.fit(X_fit, y_fit)
        X_fit_ready = preprocessor.transform(X_fit)
        X_early_ready = preprocessor.transform(X_early)
        X_outer_val_ready = preprocessor.transform(X_outer_val)
        model.fit(
            X_fit_ready,
            y_fit,
            eval_set=[(X_early_ready, y_early)],
            verbose=False,
        )
        fold_predictions = model.predict(X_outer_val_ready).astype(int)
        fold_macro_f1 = f1_score(
            y_outer_val,
            fold_predictions,
            labels=list(range(16)),
            average='macro',
            zero_division=0,
        )
        best_iteration = int(model.best_iteration)
        eval_history = model.evals_result()['validation_0'][EARLY_STOPPING_METRIC]
        rows.append({
            'fold': fold_number,
            'macro_f1': fold_macro_f1,
            'best_iteration': best_iteration,
            'n_estimators_selected': best_iteration + 1,
            'early_stopping_metric': EARLY_STOPPING_METRIC,
            'early_stopping_value_at_best': float(eval_history[best_iteration]),
        })
        if record_oof:
            oof_predictions[outer_val_pos] = fold_predictions

    if record_oof:
        assert (oof_predictions >= 0).all(), 'Nem todos os exemplos receberam previsão OOF.'
    return rows, oof_predictions

grid_results = []
for min_child_weight in [1, 5]:
    for subsample in [0.8, 1.0]:
        for colsample_bytree in [0.8, 1.0]:
            candidate_params = {
                'min_child_weight': min_child_weight,
                'subsample': subsample,
                'colsample_bytree': colsample_bytree,
            }
            fold_rows, _ = run_xgb_fold_cv(X_train, y_train, FOLD_SPLITS, candidate_params)
            fold_scores = [row['macro_f1'] for row in fold_rows]
            grid_results.append({
                **candidate_params,
                'cv_macro_f1_mean': float(np.mean(fold_scores)),
                'cv_macro_f1_std': float(np.std(fold_scores, ddof=0)),
                'fold_macro_f1': fold_scores,
                'fold_best_iteration': [row['best_iteration'] for row in fold_rows],
                'fold_n_estimators': [row['n_estimators_selected'] for row in fold_rows],
                'fold_early_stopping_metric': [row['early_stopping_metric'] for row in fold_rows],
                'fold_early_stopping_value': [row['early_stopping_value_at_best'] for row in fold_rows],
            })

grid_table = pd.DataFrame(grid_results).sort_values(
    ['cv_macro_f1_mean', 'min_child_weight', 'subsample', 'colsample_bytree'],
    ascending=[False, True, False, False],
).reset_index(drop=True)
display(grid_table)
best_row = grid_table.iloc[0]
selected_params = {
    'min_child_weight': int(best_row['min_child_weight']),
    'subsample': float(best_row['subsample']),
    'colsample_bytree': float(best_row['colsample_bytree']),
}
selected_fold_tree_counts = [int(value) for value in best_row['fold_n_estimators']]
selected_n_estimators = int(np.median(selected_fold_tree_counts))
selected_cv_mean = float(best_row['cv_macro_f1_mean'])
selected_cv_std = float(best_row['cv_macro_f1_std'])
print(f'Configuração selecionada: {selected_params}')
print(f'Macro F1 CV média/desvio padrão: {selected_cv_mean:.4f} / {selected_cv_std:.4f}')
print(f'Métrica de early stopping: {EARLY_STOPPING_METRIC}; rondas por fold: {selected_fold_tree_counts}')
print(f'Número de árvores usado no reajuste: mediana(best_iteration + 1) = {selected_n_estimators}')

,min_child_weight,subsample,colsample_bytree,cv_macro_f1_mean,cv_macro_f1_std,fold_macro_f1,fold_best_iteration,fold_n_estimators,fold_early_stopping_metric,fold_early_stopping_value
0,5,0.8,0.8,0.436522,0.003318,"[0.44003644308059947, 0.4374559689300937, 0.43...","[126, 123, 99]","[127, 124, 100]","[mlogloss, mlogloss, mlogloss]","[1.3116275093187935, 1.3101116054635091, 1.294..."
1,1,0.8,0.8,0.436175,0.002739,"[0.43236250253476904, 0.4386732448061985, 0.43...","[130, 101, 97]","[131, 102, 98]","[mlogloss, mlogloss, mlogloss]","[1.3035872449956096, 1.3192768118056462, 1.301..."
2,1,1.0,1.0,0.436128,0.006702,"[0.4266506869042173, 0.44078908710963494, 0.44...","[98, 104, 88]","[99, 105, 89]","[mlogloss, mlogloss, mlogloss]","[1.3172006319800527, 1.3319467216742862, 1.301..."
3,5,0.8,1.0,0.435661,0.006263,"[0.43006898309828107, 0.43250845821627415, 0.4...","[78, 106, 132]","[79, 107, 133]","[mlogloss, mlogloss, mlogloss]","[1.3155965752126553, 1.3305142020409504, 1.296..."
4,1,0.8,1.0,0.435012,0.003173,"[0.43055339710322016, 0.43680595093480934, 0.4...","[98, 123, 108]","[99, 124, 109]","[mlogloss, mlogloss, mlogloss]","[1.3199984614536624, 1.329913217899312, 1.3034..."
5,5,1.0,0.8,0.434551,0.007093,"[0.42693179820186405, 0.43270930930047335, 0.4...","[100, 119, 90]","[101, 120, 91]","[mlogloss, mlogloss, mlogloss]","[1.3036705119261347, 1.3143749895973924, 1.298..."
6,1,1.0,0.8,0.430609,0.007936,"[0.41992558634983457, 0.43297161618060376, 0.4...","[96, 135, 86]","[97, 136, 87]","[mlogloss, mlogloss, mlogloss]","[1.2960308334411057, 1.319363329372663, 1.3055..."
7,5,1.0,1.0,0.430507,0.000724,"[0.42986320394431327, 0.4301405311396531, 0.43...","[90, 140, 91]","[91, 141, 92]","[mlogloss, mlogloss, mlogloss]","[1.3129370672163154, 1.310922524290735, 1.3034..."


Configuração selecionada: {'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8}
Macro F1 CV média/desvio padrão: 0.4365 / 0.0033
Métrica de early stopping: mlogloss; rondas por fold: [127, 124, 100]
Número de árvores usado no reajuste: mediana(best_iteration + 1) = 124


## Reajuste e avaliação na validação fixa

Usamos a mediana dos números de árvores dos três folds selecionados, contando `best_iteration + 1`. Com esse número fixo, uma pipeline nova ajusta o pré-processamento e o XGBoost em todo `X_train` e avalia uma vez em `X_val`. O mapeamento de escalões usado no projeto coloca 30–34 na classe 4 e 75–79 na classe 13 (coerente com as classes 8=50–54 e 9=55–59 indicadas no enunciado); apresentamos também o F1 dessas duas classes.

In [3]:
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

final_model_params = {
    **FIXED_XGB_PARAMS,
    **selected_params,
    'n_estimators': selected_n_estimators,
    'early_stopping_rounds': None,
}
validation_pipe = make_model_pipeline(
    XGBClassifier(**final_model_params),
    normalize_numeric=False,
)
validation_pipe.fit(X_train, y_train)
validation_predictions = validation_pipe.predict(X_val).astype(int)
validation_macro_f1 = f1_score(
    y_val,
    validation_predictions,
    labels=list(range(16)),
    average='macro',
    zero_division=0,
)
class_report = classification_report(
    y_val,
    validation_predictions,
    labels=list(range(16)),
    output_dict=True,
    zero_division=0,
)
print(f'Macro F1 validação fixa: {validation_macro_f1:.4f}')
print(classification_report(
    y_val,
    validation_predictions,
    labels=list(range(16)),
    target_names=[str(i) for i in range(16)],
    zero_division=0,
))
print('F1 classe 4 (30–34):', f"{class_report['4']['f1-score']:.4f}")
print('F1 classe 13 (75–79):', f"{class_report['13']['f1-score']:.4f}")
confusion = confusion_matrix(y_val, validation_predictions, labels=list(range(16)))
display(pd.DataFrame(confusion, index=pd.Index(range(16), name='real'), columns=pd.Index(range(16), name='prevista')))

Macro F1 validação fixa: 0.4449
              precision    recall  f1-score   support

           0       0.78      0.85      0.81        41
           1       0.39      0.21      0.27        34
           2       0.58      0.65      0.61        80
           3       0.37      0.35      0.36        75
           4       0.31      0.22      0.26        90
           5       0.28      0.33      0.31       129
           6       0.37      0.34      0.36       172
           7       0.39      0.39      0.39       173
           8       0.58      0.66      0.61       181
           9       0.62      0.64      0.63       203
          10       0.56      0.67      0.61       213
          11       0.54      0.48      0.51       208
          12       0.40      0.47      0.44       200
          13       0.26      0.22      0.24       165
          14       0.32      0.30      0.31       130
          15       0.47      0.35      0.40        99

    accuracy                           0.46     

prevista,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
real,,,,,,,,,,,,,,,,
0,35,3,2,1,0,0,0,0,0,0,0,0,0,0,0,0
1,8,7,19,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,5,52,8,5,3,3,2,1,0,0,0,0,0,0,0
3,1,3,7,26,12,11,7,6,2,0,0,0,0,0,0,0
4,0,0,4,16,20,29,8,10,3,0,0,0,0,0,0,0
5,0,0,2,14,16,43,34,12,8,0,0,0,0,0,0,0
6,0,0,3,3,7,45,59,33,22,0,0,0,0,0,0,0
7,0,0,0,2,4,11,38,67,51,0,0,0,0,0,0,0
8,0,0,1,0,1,9,11,40,119,0,0,0,0,0,0,0


## Sensibilidade OOF com e sem perguntas condicionais

Comparamos as mesmas previsões out-of-fold nos mesmos três folds usando a configuração selecionada e o mesmo número fixo de árvores escolhido acima. A segunda versão remove apenas `PC1B`, `HA1A`, `IN15` e `IN22`; `SEX` permanece. Em cada fold e versão, o pré-processador ajusta-se apenas no treino do fold. Definimos como sinalizado o caso com previsão OOF diferente do `AGE_RANGE` observado; este é um indicador de erro do classificador, não uma identificação validada de anomalia. Comparamos interseção e diferenças dos conjuntos sinalizados e também mudanças diretas de previsão.

In [4]:
CONDITIONAL_COLUMNS = ['PC1B', 'HA1A', 'IN15', 'IN22']
missing_conditional = [column for column in CONDITIONAL_COLUMNS if column not in X_train.columns]
if missing_conditional:
    raise ValueError(f'Não é possível executar a ablação; faltam colunas: {missing_conditional}')

X_without_conditional = X_train.drop(columns=CONDITIONAL_COLUMNS)
assert 'SEX' in X_without_conditional.columns, 'SEX tem de permanecer na versão reduzida.'
oof_model_params = {
    'min_child_weight': selected_params['min_child_weight'],
    'subsample': selected_params['subsample'],
    'colsample_bytree': selected_params['colsample_bytree'],
}
oof_all_rows, oof_all = run_xgb_fold_cv(
    X_train, y_train, FOLD_SPLITS, oof_model_params, record_oof=True
)
oof_reduced_rows, oof_reduced = run_xgb_fold_cv(
    X_without_conditional, y_train, FOLD_SPLITS, oof_model_params, record_oof=True
)

oof_all_macro_f1 = f1_score(y_train, oof_all, labels=list(range(16)), average='macro', zero_division=0)
oof_reduced_macro_f1 = f1_score(y_train, oof_reduced, labels=list(range(16)), average='macro', zero_division=0)
oof_metrics = pd.DataFrame([
    {
        'versao': 'Todas as variáveis',
        'macro_f1_oof': oof_all_macro_f1,
        'fold_macro_f1': [row['macro_f1'] for row in oof_all_rows],
        'fold_n_estimators_early_stopping': [row['n_estimators_selected'] for row in oof_all_rows],
    },
    {
        'versao': 'Sem PC1B, HA1A, IN15, IN22 (SEX mantida)',
        'macro_f1_oof': oof_reduced_macro_f1,
        'fold_macro_f1': [row['macro_f1'] for row in oof_reduced_rows],
        'fold_n_estimators_early_stopping': [row['n_estimators_selected'] for row in oof_reduced_rows],
    },
])
display(oof_metrics)

y_array = y_train.to_numpy()
flagged_all = oof_all != y_array
flagged_reduced = oof_reduced != y_array
both_flagged = flagged_all & flagged_reduced
all_only = flagged_all & ~flagged_reduced
reduced_only = ~flagged_all & flagged_reduced
neither_flagged = ~flagged_all & ~flagged_reduced
changed_prediction = oof_all != oof_reduced
flag_summary = pd.DataFrame([
    {'comparacao': 'Sinalizado em ambas (interseção)', 'casos': int(both_flagged.sum())},
    {'comparacao': 'Só com todas as variáveis', 'casos': int(all_only.sum())},
    {'comparacao': 'Só sem as quatro perguntas', 'casos': int(reduced_only.sum())},
    {'comparacao': 'Não sinalizado em nenhuma', 'casos': int(neither_flagged.sum())},
    {'comparacao': 'Total com sinalização diferente', 'casos': int((all_only | reduced_only).sum())},
    {'comparacao': 'Total com classe prevista diferente', 'casos': int(changed_prediction.sum())},
])
display(flag_summary)

oof_comparison = pd.DataFrame({
    'indice_original': X_train.index,
    'AGE_RANGE_real': y_array,
    'previsto_todas': oof_all,
    'previsto_sem_condicionais': oof_reduced,
    'sinalizado_todas': flagged_all,
    'sinalizado_sem_condicionais': flagged_reduced,
})
changed_cases = oof_comparison.loc[
    oof_comparison['sinalizado_todas'] != oof_comparison['sinalizado_sem_condicionais']
].copy()
display(changed_cases.head(20))

,versao,macro_f1_oof,fold_macro_f1,fold_n_estimators_early_stopping
0,Todas as variáveis,0.437523,"[0.44003644308059947, 0.4374559689300937, 0.43...","[127, 124, 100]"
1,"Sem PC1B, HA1A, IN15, IN22 (SEX mantida)",0.437310,"[0.4328537720392221, 0.4391898932265625, 0.438...","[129, 111, 112]"


,comparacao,casos
0,Sinalizado em ambas (interseção),4242
1,Só com todas as variáveis,559
2,Só sem as quatro perguntas,552
3,Não sinalizado em nenhuma,3416
4,Total com sinalização diferente,1111
5,Total com classe prevista diferente,1705


,indice_original,AGE_RANGE_real,previsto_todas,previsto_sem_condicionais,sinalizado_todas,sinalizado_sem_condicionais
2,1698,10,10,9,False,True
7,3090,0,2,0,True,False
12,2192,11,11,12,False,True
14,1463,12,12,9,False,True
24,2021,9,10,9,True,False
29,10313,13,12,13,True,False
38,6231,13,13,14,False,True
40,10898,14,13,14,True,False
62,3518,8,8,7,False,True
63,9747,11,12,11,True,False


## Conclusão e decisão
- Data: 9/11

A configuração escolhida foi `max_depth=5`, `learning_rate=0.1`, `min_child_weight=5`, `subsample=0.8` e `colsample_bytree=0.8`. Com early stopping por `mlogloss`, os folds escolheram **127, 124 e 100 árvores** (`best_iteration + 1`); a mediana, **124**, foi usada no reajuste. A Macro F1 média CV foi **0,4365** (desvio padrão **0,0033**) e a validação fixa obteve **0,4449**. Face ao 007, a CV caiu **0,0084** (0,4449 → 0,4365) e a validação fixa caiu **0,0042** (0,4491 → 0,4449). O resultado continua acima do 002, 005 e 006. O F1 foi **0,2581** na classe 4 (30–34) e **0,2410** na classe 13 (75–79), pelo que estes escalões continuam difíceis.

Na análise OOF, a Macro F1 foi **0,4375** com todas as variáveis e **0,4373** sem `PC1B`, `HA1A`, `IN15` e `IN22` (diferença +0,0002). **4.242** casos foram sinalizados nas duas versões; **559** apenas com todas as variáveis e **552** apenas sem as quatro perguntas, totalizando **1.111** casos com estado de sinalização diferente. As classes previstas mudaram em **1.705** casos. Portanto, scores agregados quase iguais escondem mudanças em indivíduos específicos; o sinal usado aqui é erro OOF e não verdade de anomalia.

A configuração ajustada não melhorou o 007 na validação fixa; manter o 007 como candidato atual e considerar uma nova submissão apenas se houver motivo para medir a diferença no Kaggle.
- Sem submissão 